# Tutorial on column generation for cutting stock problem with JuMP

https://jump.dev/JuMP.jl/stable/tutorials/algorithms/cutting_stock_column_generation/

In [3]:
using Pkg
Pkg.activate("../")

  Activating project at `c:\Users\yshimane3\Documents\codes\dev-julia\TelescopeTasking.jl`


In [25]:
using JuMP
import DataFrames
import HiGHS
using GLMakie
import SparseArrays

## Background data

In [12]:
struct Piece
    w::Float64
    d::Int
end

struct Data
    pieces::Vector{Piece}
    W::Float64
end

function Base.show(io::IO, d::Data)
    println(io, "Data for the cutting stock problem:")
    println(io, "  W = $(d.W)")
    println(io, "with pieces:")
    println(io, "   i   w_i d_i")
    println(io, "  ------------")
    for (i, p) in enumerate(d.pieces)
        println(io, lpad(i, 4), " ", lpad(p.w, 5), " ", lpad(p.d, 3))
    end
    return
end

function get_data()
    data = [
        75.0 38
        75.0 44
        75.0 30
        75.0 41
        75.0 36
        53.8 33
        53.0 36
        51.0 41
        50.2 35
        32.2 37
        30.8 44
        29.8 49
        20.1 37
        16.2 36
        14.5 42
        11.0 33
        8.6 47
        8.2 35
        6.6 49
        5.1 42
    ]
    return Data([Piece(data[i, 1], data[i, 2]) for i in axes(data, 1)], 100.0)
end

data = get_data()

Data for the cutting stock problem:
  W = 100.0
with pieces:
   i   w_i d_i
  ------------
   1  75.0  38
   2  75.0  44
   3  75.0  30
   4  75.0  41
   5  75.0  36
   6  53.8  33
   7  53.0  36
   8  51.0  41
   9  50.2  35
  10  32.2  37
  11  30.8  44
  12  29.8  49
  13  20.1  37
  14  16.2  36
  15  14.5  42
  16  11.0  33
  17   8.6  47
  18   8.2  35
  19   6.6  49
  20   5.1  42


In [14]:
I = length(data.pieces)
J = 1_000  # Some large number

1000

## Solve via column generation

For the initial set of patterns, we create a trivial cutting pattern which cuts as many units of piece i
 as will fit.

In [15]:
patterns = map(1:I) do i
    n_pieces = floor(Int, data.W / data.pieces[i].w)
    return SparseArrays.sparsevec([i], [n_pieces], I)
end

20-element Vector{SparseArrays.SparseVector{Int64, Int64}}:
   [1]  =  1
   [2]  =  1
   [3]  =  1
   [4]  =  1
   [5]  =  1
   [6]  =  1
   [7]  =  1
   [8]  =  1
   [9]  =  1
   [10]  =  3
   [11]  =  3
   [12]  =  3
   [13]  =  4
   [14]  =  6
   [15]  =  6
   [16]  =  9
   [17]  =  11
   [18]  =  12
   [19]  =  15
   [20]  =  19

In [35]:
"""
    cutting_locations(data::Data, pattern::SparseArrays.SparseVector)

A function which returns a vector of the locations along the roll at which to
cut in order to produce pattern `pattern`.
"""
function cutting_locations(data::Data, pattern::SparseArrays.SparseVector)
    locations = Float64[]
    offset = 0.0
    for (i, c) in zip(SparseArrays.findnz(pattern)...)
        for _ in 1:c
            offset += data.pieces[i].w
            push!(locations, offset)
        end
    end
    return locations
end

function plot_patterns(data::Data, patterns)
    fig = Figure()
    ax = Axis(fig[1, 1]; xlabel = "Pattern", ylabel = "Roll length")
    xlims!(ax, 0, length(patterns) + 1)
    ylims!(ax, 0, data.W)
    
    for (i, p) in enumerate(patterns)
        locations = cutting_locations(data, p)
        barplot!(
            ax,
            fill(i, length(locations)),
            reverse(locations);
            color = :gray85, strokecolor = :black, strokewidth = 0.5
            # bar_width = 0.6,
            # label = false,
            # color = "#90caf9",
        )
    end
    return fig
end

plot_patterns (generic function with 1 method)

In [36]:
fig = plot_patterns(data, patterns)
display(fig)

GLMakie.Screen(...)

## Base problem

This only uses the initial set of patterns

In [37]:
model = Model(HiGHS.Optimizer)
set_silent(model)
@variable(model, x[1:length(patterns)] >= 0, Int)
@objective(model, Min, sum(x))
@constraint(model, demand[i in 1:I], patterns[i]' * x >= data.pieces[i].d)
optimize!(model)
@assert is_solved_and_feasible(model)
solution_summary(model)

* Solver : HiGHS

* Status
  Result count       : 1
  Termination status : OPTIMAL
  Message from the solver:
  "kHighsModelStatusOptimal"

* Candidate solution (result #1)
  Primal status      : FEASIBLE_POINT
  Dual status        : NO_SOLUTION
  Objective value    : 4.21000e+02
  Objective bound    : 4.21000e+02
  Relative gap       : 0.00000e+00

* Work counters
  Solve time (sec)   : 4.10199e-03
  Simplex iterations : 0
  Barrier iterations : -1
  Node count         : 0


## Choosing new columns

In [41]:
unset_integer.(x)
optimize!(model)
@assert is_solved_and_feasible(model; dual = true)
π_13 = dual(demand[13])

0.25

In [42]:
function solve_pricing(data::Data, π::Vector{Float64})
    I = length(π)
    model = Model(HiGHS.Optimizer)
    set_silent(model)
    @variable(model, y[1:I] >= 0, Int)
    @constraint(model, sum(data.pieces[i].w * y[i] for i in 1:I) <= data.W)
    @objective(model, Max, sum(π[i] * y[i] for i in 1:I))
    optimize!(model)
    @assert is_solved_and_feasible(model)
    number_of_rolls_saved = objective_value(model)
    if number_of_rolls_saved > 1 + 1e-8
        # Benefit of pattern is more than the cost of a new roll plus some
        # tolerance
        return SparseArrays.sparse(round.(Int, value.(y)))
    end
    return nothing
end

solve_pricing (generic function with 1 method)

In [47]:
solve_pricing(data, [1.0 / i for i in 1:I])

20-element SparseArrays.SparseVector{Int64, Int64} with 3 stored entries:
  [1 ]  =  1
  [17]  =  1
  [20]  =  3

In [50]:
solve_pricing(data, zeros(I))

## Iterative algorithm

In [51]:
while true
    # Solve the linear relaxation
    optimize!(model)
    @assert is_solved_and_feasible(model; dual = true)
    # Obtain a new dual vector
    π = dual.(demand)
    # Solve the pricing problem
    new_pattern = solve_pricing(data, π)
    # Stop iterating if there is no new pattern
    if new_pattern === nothing
        @info "No new patterns, terminating the algorithm."
        break
    end
    push!(patterns, new_pattern)
    # Create a new column
    push!(x, @variable(model, lower_bound = 0))
    # Update the objective coefficient of the new column
    set_objective_coefficient(model, x[end], 1.0)
    # Update the non-zeros in the coefficient matrix
    for (i, count) in zip(SparseArrays.findnz(new_pattern)...)
        set_normalized_coefficient(demand[i], x[end], count)
    end
    println("Found new pattern. Total patterns = $(length(patterns))")
end

Found new pattern. Total patterns = 21
Found new pattern. Total patterns = 22
Found new pattern. Total patterns = 23
Found new pattern. Total patterns = 24
Found new pattern. Total patterns = 25
Found new pattern. Total patterns = 26
Found new pattern. Total patterns = 27
Found new pattern. Total patterns = 28
Found new pattern. Total patterns = 29
Found new pattern. Total patterns = 30
Found new pattern. Total patterns = 31
Found new pattern. Total patterns = 32
Found new pattern. Total patterns = 33
Found new pattern. Total patterns = 34
Found new pattern. Total patterns = 35


┌ Info: No new patterns, terminating the algorithm.
└ @ Main c:\Users\yshimane3\Documents\codes\dev-julia\TelescopeTasking.jl\dev\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X25sZmlsZQ==.jl:11


In [52]:
plot_patterns(data, patterns)